In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
import numpy as np
import fiona # enging for reading more robust data files
import contextily as cx
import os
from matplotlib import pyplot as plot

In [2]:
# first ensure you are in the right directory!
#os.chdir("/users/username/MinGenResources-DFW/Parks")

In [3]:
# 2020 block group data from the U.S. Census
#block_groups = gpd.read_file("tl_2020_48_bg.shp", engine="fiona")

In [4]:
# select only the block groups in Dallas County, code 113
#dallas_county_block_groups = block_groups[block_groups['COUNTYFP']=='113']
# save Dallas County block groups as a new file
#dallas_county_block_groups.to_file("dallas_county_block_groups.geojson")

In [5]:
# read in block group geometries for Dallas County
dallas_county_block_groups = gpd.read_file("data/demographic_data/dallas_county_block_groups.geojson")

In [6]:
# rename column to match demographic data so that we can merge
dallas_county_block_groups = dallas_county_block_groups.rename(columns={"GEOID": "GEO_ID"})

In [7]:
# only select the columns needed
dallas_county_block_groups = dallas_county_block_groups[["GEO_ID", "INTPTLAT", "INTPTLON", "geometry"]]

Age of Population

In [8]:
# read in age data
age = pd.read_csv("data/demographic_data/age.csv")
# only take the digits of GEO_ID so that it matches the format of block group data for merging
age["GEO_ID"] = age["GEO_ID"].str[9:]
# only select the columns we are interested in
# P15_001N -> total individuals
# P15_002N -> total under 18
# P15_003N -> total 18 and above
age = age[["GEO_ID", "P15_001N", "P15_002N", "P15_003N"]]

In [9]:
# rename columns for readability
age = age.rename(columns={"P15_001N": "Total", "P15_002N": "Children", "P15_003N": "Adults"})
# drop the first row which tells us what the column names mean
age = age.drop(labels = 0, axis = 0)

In [10]:
# merge age and block group data on the GEO ID to create a new data frame 
geo_age = dallas_county_block_groups.merge(age, on="GEO_ID", how="left")

In [11]:
# These are currently not numeric data types, need to typecast them
geo_age["Total"] = pd.to_numeric(geo_age["Total"])
geo_age["Children"] = pd.to_numeric(geo_age["Children"])
geo_age["Adults"] = pd.to_numeric(geo_age["Adults"])

In [12]:
# create a column called percent_child to indicate what percentage of each block group is under 18
geo_age["percent_child"] = geo_age["Children"]/geo_age["Total"]*100

In [13]:
# geo_age.isna().sum()
# geo_age[geo_age["percent_child"].isna()]
# We have a few NAs in percent_child due to division by 0 - where there are 0 people in the block group
geo_age = geo_age.replace(np.NaN, 0)
#geo_age[geo_age["percent_child"].isna()]

In [14]:
# change to crs used for mapping
geo_age = geo_age.to_crs("EPSG:3857")

In [15]:
# Plot to check
# ax = geo_age.plot(column = "percent_child", legend = True)
# cx.add_basemap(ax)
# ax.set_title("Percent of Population under 18")
# ax.set_axis_off();

In [16]:
# Plot to check
# ax = geo_age.plot(column = "Children", legend = True)
# cx.add_basemap(ax)
# ax.set_title("Number of Children")
# ax.set_axis_off();

Median Income

In [17]:
# read in income data
income = pd.read_csv("data/demographic_data/median_income.csv")

In [18]:
# only take the digits of GEO_ID so that it matches the format of block group data for merging
income["GEO_ID"] = income["GEO_ID"].str[9:]

In [19]:
# only select the columns we are interested in
# B19013_001E -> Median Household Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)
income = income[["GEO_ID", "B19013_001E"]]

In [20]:
# rename the column for readability
income = income.rename(columns={"B19013_001E": "Median_Income"})
# drop the first row which tells us what the column names mean
income = income.drop(labels = 0, axis = 0)

In [21]:
# merge income and block group data on the GEO ID to create a new data frame 
geo_income = dallas_county_block_groups.merge(income, on="GEO_ID", how="left")

In [22]:
#geo_income["Median_Income"].value_counts()
# replace the dashes with NA
geo_income = geo_income.replace('-', pd.NA)
# remove the comma and the plus sign, represent 250,000 as just the minimum 250000
geo_income = geo_income.replace('250,000+', "250000")
#geo_income["Median_Income"].value_counts()

In [23]:
# these are currently not numeric data types, need to typecast them
geo_income["Median_Income"] = pd.to_numeric(geo_income["Median_Income"])

In [24]:
# change to crs used for mapping
geo_income = geo_income.to_crs("EPSG:3857")

In [25]:
# There are many missing values
# geo_income.isna().sum()

In [26]:
# plot to check
# ax = geo_income.plot(column = "Median_Income", legend = True)
# cx.add_basemap(ax)
# ax.set_title("Median Income")
# ax.set_axis_off();

Poverty

In [27]:
# read in census data
poverty = pd.read_csv("data/demographic_data/poverty.csv")

In [28]:
# only take the digits of GEO_ID so that it matches the format of block group data for merging
poverty["GEO_ID"] = poverty["GEO_ID"].str[9:]

In [29]:
# B17010_001E -> Total households in the block group
# B17010_002E -> Number of households below the poverty 
# B17010_004E -> Married householders with children
# B17010_011E -> Male householder with children
# B17010_017E -> Female householder with children
poverty = poverty[["GEO_ID", "B17010_001E", "B17010_002E", "B17010_004E", "B17010_011E", "B17010_017E"]]

In [30]:
# rename columns for readability
poverty = poverty.rename(columns={"B17010_001E": "households", "B17010_002E": "households_below_poverty", "B17010_004E": "married_with__children", 
                                 "B17010_011E": "male_with_children", "B17010_017E": "female_with_children"})

In [31]:
# drop the first row which tells us what the column names mean
poverty = poverty.drop(labels = 0, axis = 0)

In [32]:
# these are currently not numeric data types, need to typecast them
poverty["households"] = pd.to_numeric(poverty["households"])
poverty["households_below_poverty"] = pd.to_numeric(poverty["households_below_poverty"])
poverty["married_with__children"] = pd.to_numeric(poverty["married_with__children"])
poverty["male_with_children"] = pd.to_numeric(poverty["male_with_children"])
poverty["female_with_children"] = pd.to_numeric(poverty["female_with_children"])

In [33]:
# merge poverty and block group data on the GEO ID to create a new data frame 
geo_poverty = dallas_county_block_groups.merge(poverty, on="GEO_ID", how="left")

In [34]:
# change to crs used for mapping
geo_poverty = geo_poverty.to_crs("EPSG:3857")

In [35]:
# create a column to indicate what percentage of households in the block group are below the poverty level
geo_poverty["percent_below"] = geo_poverty["households_below_poverty"]/geo_poverty["households"]*100

In [36]:
# create a column to indicate the total number of households with children in each block group
geo_poverty["households_with_children"] = (geo_poverty["married_with__children"]+
                                geo_poverty["male_with_children"]+
                                geo_poverty["female_with_children"])

In [37]:
# create a column to indicate the percentage households in each block group who are below the poverty level and have children
geo_poverty["child_poverty_percent"] = geo_poverty["households_with_children"]/geo_poverty["households"]*100

In [38]:
# some Na's from division by zero
#geo_poverty[geo_poverty["percent_below"].isna()]
#geo_poverty[geo_poverty["child_poverty_percent"].isna()]
geo_poverty = geo_poverty.replace(np.NaN, 0)
#geo_poverty[geo_poverty["child_poverty_percent"].isna()]
#geo_poverty[geo_poverty["percent_below"].isna()]

In [39]:
# write the merged data files as new files
#geo_income.to_file("geo_income.geojson")
#geo_age.to_file("geo_age.geojson")
#geo_poverty.to_file("geo_poverty.geojson")